# Import library

In [1]:
import numpy as np
import pandas as pd
import torch
from fastai.learner import load_learner
from pathlib import Path

In [2]:
# Allow full column width display in Pandas
pd.set_option('display.max_colwidth', None)

# Load all model predictions files

## Resnet 101
- Notebook link: https://www.kaggle.com/code/limdx006/cp3501-2025-task-4-group-20-lim-resnet101
- Final result in task 4: 1.4031

In [3]:
resnet101_model = load_learner("/kaggle/input/cp3501-2025-task-4-group-20-lim-resnet101/resnet101_model.pkl", cpu=False)

/usr/local/lib/python3.11/dist-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


## ResNet34
- Notebook link: https://www.kaggle.com/code/richo7777/cp3501-2025-task-4-group-20-resnet34
- Final result in task 4: 1.5970

In [4]:
resnet34_model = load_learner("/kaggle/input/cp3501-2025-task-4-group-20-resnet34/resnet34_model.pkl", cpu=False)

/usr/local/lib/python3.11/dist-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


## DenseNet121
- Notebook link: https://www.kaggle.com/code/sinnatherpaing/cp3501-2025-task-4-6p-group-20
- Final result in task 4: 1.4956

In [5]:
densenet121_model = load_learner("/kaggle/input/cp3501-2025-task-4-6p-group-20/densenet121_model_6p.pkl", cpu=False)

## ResNet50
- Notebook link: https://www.kaggle.com/code/alexiawin/cp3501-2025-task3-resnet50augmentation-labelsmooth#Submit-to-Kaggle-&-Record-Public-Score
- Final result in task 4: 1.6262

In [6]:
resnet50_model = load_learner("/kaggle/input/cp3501-2025-task3-resnet50augmentation-labelsmooth/resnet50_preds.pkl", cpu=False)

## beit_base_patch16_224
- Notebook link: https://www.kaggle.com/code/choopwintchal/cp3501-2025-task-4-group-20-2nd-model/notebook
- Final result in task 4: 1.2476

In [7]:
beit224_model = load_learner("/kaggle/input/cp3501-2025-task-4-group-20-2nd-model/beit_base_patch16_224_Aug_TTA_LR.pkl", cpu=False)

# Get test predictions

## Load test data

In [8]:
# Read the CSV
test_df = pd.read_csv('/kaggle/input/competition-files/test_features.csv')
test_df.head()

,id,filepath,site
0,ZJ016488,test_features/ZJ016488.jpg,S0082
1,ZJ016489,test_features/ZJ016489.jpg,S0040
2,ZJ016490,test_features/ZJ016490.jpg,S0040
3,ZJ016491,test_features/ZJ016491.jpg,S0041
4,ZJ016492,test_features/ZJ016492.jpg,S0040


In [9]:
# Get list of image paths
test_dir = Path('/kaggle/input/competition-files/') 
test_files = [test_dir/f for f in test_df['filepath']]
print("Loaded image count:", len(test_files))     # should be 4464

Loaded image count: 4464


## Get predictions (TTA) for all model
- Use the model each datablock for the prediction

In [10]:
print("101 Predicting......") # Add output for debugging as it was taking longer than I though
# tta_preds101, _ = resnet101_model.tta(dl=resnet101_model.dls.test_dl(test_files))
print("101 Done!")

101 Predicting......
101 Done!


In [11]:
print("34 Predicting......") 
tta_preds34, _ = resnet34_model.tta(dl=resnet34_model.dls.test_dl(test_files))
print("34 Done!")

34 Predicting......


34 Done!


In [12]:
print("121 Predicting......")
tta_preds121, _ = densenet121_model.tta(dl=densenet121_model.dls.test_dl(test_files))
print("121 Done!")

121 Predicting......


121 Done!


In [13]:
# Manually change the model to GPU as CPU=false is not working
resnet50_model.model.to('cuda:0')
resnet50_model.dls.device = torch.device('cuda:0')

print("50 Predicting......") 
tta_preds50, _ = resnet50_model.tta(dl=resnet50_model.dls.test_dl(test_files))
print("50 Done!")

50 Predicting......


50 Done!


In [14]:
print("224 Predicting......") 
# tta_preds224, _ = beit224_model.tta(dl=beit224_model.dls.test_dl(test_files))
print("224 Done!")

224 Predicting......
224 Done!


# Average predictions
- Try both medium and mean to compare result

In [15]:
# Gather all predictions result
# all_preds = [tta_preds101, tta_preds34, tta_preds121, tta_preds50, tta_preds224]
# all_preds = [tta_preds101, tta_preds121, tta_preds224]
all_preds = [tta_preds34, tta_preds121, tta_preds50]

In [16]:
# Stack Predictions
print("Doing stacked preds......")
stacked_preds = torch.stack(all_preds)
print("Stacked preds done!")

Doing stacked preds......
Stacked preds done!


In [17]:
# Mean ensemble
ensemble_preds_mean = stacked_preds.mean(0)

In [18]:
# Median ensemble
ensemble_preds_median, _ = stacked_preds.median(0)

In [19]:
# For observe and compare
print("Mean preds:\n", ensemble_preds_mean[:5])
print("Median preds:\n", ensemble_preds_median[:5])

Mean preds:
 tensor([[0.1023, 0.0182, 0.1587, 0.3312, 0.0155, 0.2988, 0.0239, 0.0516],
        [0.4219, 0.1757, 0.0509, 0.0552, 0.0496, 0.0266, 0.1670, 0.0530],
        [0.3350, 0.0617, 0.0555, 0.2961, 0.0845, 0.0382, 0.0767, 0.0523],
        [0.0147, 0.0149, 0.0177, 0.0450, 0.0262, 0.8513, 0.0126, 0.0175],
        [0.3091, 0.1244, 0.0525, 0.0608, 0.0459, 0.0134, 0.2461, 0.1479]])
Median preds:
 tensor([[0.1033, 0.0174, 0.1093, 0.3528, 0.0168, 0.3430, 0.0241, 0.0582],
        [0.4209, 0.1504, 0.0512, 0.0598, 0.0570, 0.0308, 0.1471, 0.0541],
        [0.2655, 0.0531, 0.0477, 0.2096, 0.0847, 0.0340, 0.0717, 0.0559],
        [0.0197, 0.0195, 0.0101, 0.0380, 0.0274, 0.8261, 0.0162, 0.0219],
        [0.3130, 0.1230, 0.0572, 0.0431, 0.0348, 0.0133, 0.2042, 0.0867]])


# Save Submission

In [20]:
submission_template = pd.read_csv('/kaggle/input/competition-files/submission_format.csv')

In [21]:
# Get the class columns (all except 'id')
class_cols = submission_template.columns[1:]

In [22]:
# Prepare mean predictions (convert tensor to numpy if needed)
preds_mean_np = ensemble_preds_mean.numpy() if hasattr(ensemble_preds_mean, 'numpy') else ensemble_preds_mean

In [23]:
# Prepare median predictions (convert tensor to numpy if needed)
preds_median_np = ensemble_preds_median.numpy() if hasattr(ensemble_preds_median, 'numpy') else ensemble_preds_median

## Mean submission

In [24]:
# Create submission DataFrame for mean predictions
submission_mean = submission_template.copy()
submission_mean[class_cols] = preds_mean_np
submission_mean.to_csv('submission_mean.csv', index=False)

# Print first few rows to verify
print("Mean Submission Sample:")
print(pd.read_csv('submission_mean.csv').head())

Mean Submission Sample:
         id  antelope_duiker      bird     blank  civet_genet       hog  \
0  ZJ016488         0.102319  0.018163  0.158657     0.331167  0.015496   
1  ZJ016489         0.421871  0.175746  0.050926     0.055229  0.049585   
2  ZJ016490         0.334998  0.061679  0.055495     0.296102  0.084502   
3  ZJ016491         0.014708  0.014942  0.017687     0.045036  0.026220   
4  ZJ016492         0.309135  0.124370  0.052495     0.060759  0.045892   

    leopard  monkey_prosimian    rodent  
0  0.298775          0.023858  0.051566  
1  0.026577          0.167026  0.053039  
2  0.038219          0.076679  0.052327  
3  0.851323          0.012614  0.017469  
4  0.013380          0.246096  0.147873  


## Median submission

In [25]:
# Create submission DataFrame for median predictions
submission_median = submission_template.copy()
submission_median[class_cols] = preds_median_np
submission_median.to_csv('submission_median.csv', index=False)

# Print first few rows to verify
print("\nMedian Submission Sample:")
print(pd.read_csv('submission_median.csv').head())


Median Submission Sample:
         id  antelope_duiker      bird     blank  civet_genet       hog  \
0  ZJ016488         0.103284  0.017436  0.109346     0.352768  0.016842   
1  ZJ016489         0.420858  0.150414  0.051183     0.059825  0.056970   
2  ZJ016490         0.265486  0.053056  0.047678     0.209590  0.084733   
3  ZJ016491         0.019726  0.019516  0.010117     0.037975  0.027400   
4  ZJ016492         0.312993  0.123047  0.057235     0.043090  0.034767   

    leopard  monkey_prosimian    rodent  
0  0.343049          0.024150  0.058207  
1  0.030772          0.147139  0.054124  
2  0.033975          0.071659  0.055897  
3  0.826104          0.016196  0.021902  
4  0.013334          0.204175  0.086676  
